# 🏋️ AI Search + Agent Service: Fitness-Fun Example 🤸

Welcome to our **AI Search + AI Agent** tutorial, where we'll:

1. **Create** an Azure AI Search index with fitness-oriented sample data
2. **Demonstrate** search functionality with fitness equipment queries
3. **Create** an AI agent with fitness knowledge and health disclaimers
4. **Show** how to have conversations about fitness advice and equipment recommendations

## 🏥 Health & Fitness Disclaimer
> **This notebook is for general demonstration and entertainment purposes, NOT a substitute for professional medical advice.**
> Always seek the advice of certified health professionals.

## Prerequisites
1. Complete Agent basics notebook - [1-basics.ipynb](1-basics.ipynb)
2. An **Azure AI Search** resource connected to your Microsoft Foundry project.

## What You'll Learn
- ✅ **Azure AI Search**: Create indexes, upload documents, perform searches
- ✅ **Agent Service**: Create AI agents with domain-specific knowledge
- ✅ **Integration Patterns**: How to connect search functionality with AI agents
- ✅ **Best Practices**: Error handling, resource cleanup, health disclaimers

## High-Level Flow
We'll do the following:
1. **Create** an AI Search index programmatically with sample fitness data.
2. **Upload** documents (fitness items) to the index.
3. **Verify** search functionality with test queries.
4. **Create** an AI agent with fitness expertise and proper disclaimers.
5. **Test** agent conversations about fitness equipment and advice.
6. **Clean up** resources responsibly.
 
 <img src="./seq-diagrams/5-ai-search.png" width="30%"/>

## 🔐 Authentication Setup

Before running the next cell, make sure you're authenticated with Azure CLI. 

* Open a terminal inside VSC (Visual Studio Code).
    * Run the following command in your terminal:

```
az login --use-device-code
```

* This will provide you with a device code and URL to authenticate in your browser to Azure.
    * Authenticate using the skillable Azure **Username** and **TAP**(Temporary Access Pass).
* Go back to the terminal and select the **default subscription.**

The Device Token will be used in this lab for:

* Remote development environments
* Systems without a default browser
* Corporate environments with strict security policies

* After successful authentication, you can proceed with the notebook cells below.

## 📋 Prerequisite: Connect Azure AI Search

Before proceeding, connect an Azure AI Search resource to your Microsoft Foundry project.

1. Complete [Connect to Azure AI Search](../Lab%2000%20-%20Prequisite%20-%20AI%20Foundry%20Resource%20Creation/04-Connect-to-Azure-AI-Search.md) from Lab 00.
2. Confirm the connection is available in your Foundry project before running the next cell.

> You can skip these steps if the Lab 00 connection is already complete. The notebook automatically retrieves the project's default Azure AI Search connection.

## 1. Create and Populate an Azure AI Search Index

Create a uniquely named sample index and upload a few fitness catalog items. The code retrieves the endpoint and key from the default Azure AI Search connection in your Microsoft Foundry project, so no additional search connection variables are required.

In [ ]:
import os
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchableField,
    SearchFieldDataType,
    SearchIndex,
    SimpleField,
)
from azure.identity import AzureCliCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    AISearchIndexResource,
    AzureAISearchQueryType,
    AzureAISearchTool,
    AzureAISearchToolResource,
    ConnectionType,
    PromptAgentDefinition,
)

env_path = next(
    (directory / ".env" for directory in (Path.cwd(), *Path.cwd().parents) if (directory / ".env").is_file()),
    None,
)
if env_path is None:
    raise FileNotFoundError(
        "Could not find .env. Complete Lab 00 and place it in the repository root."
    )

load_dotenv(env_path)
tenant_id = os.environ.get("TENANT_ID")
ai_foundry_project_endpoint = os.environ.get("AI_FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.environ.get("MODEL_DEPLOYMENT_NAME")
missing_variables = [
    name
    for name, value in {
        "TENANT_ID": tenant_id,
        "AI_FOUNDRY_PROJECT_ENDPOINT": ai_foundry_project_endpoint,
        "MODEL_DEPLOYMENT_NAME": model_deployment_name,
    }.items()
    if not value
]
if missing_variables:
    raise ValueError(f"Missing required .env variables: {', '.join(missing_variables)}")

credential = AzureCliCredential(tenant_id=tenant_id)
project_client = AIProjectClient(
    endpoint=ai_foundry_project_endpoint,
    credential=credential,
)
openai_client = project_client.get_openai_client()

search_conn = project_client.connections.get_default(
    connection_type=ConnectionType.AZURE_AI_SEARCH,
    include_credentials=True,
)
search_credential = AzureKeyCredential(search_conn.credentials["key"])
index_name = f"myfitnessindex-{uuid4().hex[:8]}"

index_client = SearchIndexClient(
    endpoint=search_conn.target,
    credential=search_credential,
)
search_client = SearchClient(
    endpoint=search_conn.target,
    index_name=index_name,
    credential=search_credential,
)
print(f"📁 Environment loaded from: {env_path}")
print(f"✅ Initialized clients for index: {index_name}")

## Create Search Index Schema

In [ ]:
# Define index schema with fitness equipment fields.
fields = [
    SimpleField(name="FitnessItemID", type=SearchFieldDataType.String, key=True),
    SearchableField(name="Name", type=SearchFieldDataType.String, filterable=True),
    SearchableField(name="Category", type=SearchFieldDataType.String, filterable=True, facetable=True),
    SimpleField(name="Price", type=SearchFieldDataType.Double, filterable=True, sortable=True, facetable=True),
    SearchableField(name="Description", type=SearchFieldDataType.String),
]

index = SearchIndex(name=index_name, fields=fields)
created_index = index_client.create_index(index)
print(f"🎉 Created index: {created_index.name}")

## Upload Sample Documents

In [ ]:
# Sample fitness equipment documents
sample_docs = [
    {
        "FitnessItemID": "1",
        "Name": "Adjustable Dumbbell",
        "Category": "Strength", 
        "Price": 59.99,
        "Description": "A compact, adjustable weight for targeted muscle workouts."
    },
    {
        "FitnessItemID": "2",
        "Name": "Yoga Mat",
        "Category": "Flexibility",
        "Price": 25.0,
        "Description": "Non-slip mat designed for yoga, Pilates, and other exercises."
    },
    {
        "FitnessItemID": "3",
        "Name": "Treadmill",
        "Category": "Cardio",
        "Price": 499.0,
        "Description": "A sturdy treadmill with adjustable speed and incline settings."
    },
    {
        "FitnessItemID": "4",
        "Name": "Resistance Bands",
        "Category": "Strength",
        "Price": 15.0,
        "Description": "Set of colorful bands for light to moderate resistance workouts."
    }
]

# Upload documents to index
result = search_client.upload_documents(documents=sample_docs)
print(f"🚀 Uploaded {len(sample_docs)} documents to index")


## Verify Search Functionality

In [ ]:
# Test search with "Strength" query
results = search_client.search(search_text="Strength", top=10)

print("🔍 Search results for 'Strength':")
print("-" * 40)
for doc in results:
    print(f"Name: {doc['Name']}")
    print(f"Category: {doc['Category']}")
    print(f"Price: ${doc['Price']:.2f}")
    print(f"Description: {doc['Description']}")
    print("-" * 40)

## 2. Create Agent With Azure AI Search Tool

Configure the official AzureAISearchTool and create an AI agent with direct search integration.

In [ ]:
ai_search_tool = AzureAISearchTool(
    azure_ai_search=AzureAISearchToolResource(
        indexes=[
            AISearchIndexResource(
                project_connection_id=search_conn.id,
                index_name=index_name,
                query_type=AzureAISearchQueryType.SIMPLE,
            )
        ]
    )
)

agent = project_client.agents.create_version(
    agent_name="fitness-search-agent",
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="""You are a Fitness Shopping Assistant with access to a live
        product catalog through Azure AI Search.

        Search the catalog when users ask about products. Recommend equipment based
        on goals, budget, and safety. Include safe-use guidance and remind users that
        health information is educational, not medical advice.""",
        tools=[ai_search_tool],
    ),
)

print(f"🎉 Created agent {agent.name}, version: {agent.version}")
print(f"✅ Connected to search index: {index_name}")

## 3. Test Agent with Search Integration

Test the agent with fitness equipment questions. The agent will automatically use the Azure AI Search tool when needed.

In [ ]:
def test_agent_question(question):
    conversation = openai_client.conversations.create()
    response = openai_client.responses.create(
        conversation=conversation.id,
        input=question,
        extra_body={
            "agent_reference": {
                "name": agent.name,
                "type": "agent_reference",
            }
        },
    )
    print(f"\n🔍 Question: {question}")
    print(f"🤖 Agent Response:\n{response.output_text}")
    return conversation, response


test_questions = [
    "What strength training equipment do you have available under $100?",
    "I'm looking for cardio equipment. What do you recommend?",
    "What's the cheapest fitness equipment you have?",
]

search_agent_runs = [
    test_agent_question(question) for question in test_questions
]
print("\n✅ Agent testing completed!")

In [ ]:
for conversation, _ in search_agent_runs:
    openai_client.conversations.delete(conversation_id=conversation.id)
print("🗑️ Deleted Conversations")

project_client.agents.delete_version(
    agent_name=agent.name,
    agent_version=agent.version,
)
index_client.delete_index(index_name)
print("🗑️ Deleted agent version and search index")

search_client.close()
index_client.close()
openai_client.close()
project_client.close()
credential.close()
print("✅ Cleanup completed!")

# 🎉 Congratulations!

You preserved the sample Azure AI Search index lifecycle, configured `AzureAISearchTool` with `AzureAISearchToolResource` and `AISearchIndexResource`, created a versioned prompt agent, and queried it through the Responses API. The search connection remains a Foundry project connection while the sample index is created and deleted by this notebook.